In [1]:
import sys
import os

# Use an absolute path or a relative path
# Example: '../src' if your script is in a folder one level up
module_path = os.path.abspath(os.path.join('../utils'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
import numpy as np 
from scipy import sparse 
from matplotlib import pyplot as plt
from matplotlib import animation
from time import time

from ETD_solver import cheb

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [8]:
def animate_heat_2d(U, X, Y, title, filename, interval=50, set_vlim=True, x_lim=None, y_lim=None, z_lim=None):

    # create figure and axes
    fig = plt.figure()
    ax = fig.add_subplot(111, projection="3d")

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.set_title(title)

    if set_vlim:
        vmin = U.min()
        vmax = U.max()

    # update function
    def update(t):
        ax.clear()
        if x_lim is not None: ax.set_xlim(x_lim)
        else: ax.set_xlim([0, 1])
        if y_lim is not None: ax.set_ylim(y_lim)
        else: ax.set_ylim([0, 1])
        if z_lim is not None: ax.set_zlim(z_lim)
        else: ax.set_zlim([0, 1])
        
        if set_vlim: ax.plot_surface(X, Y, U[t], cmap="magma", vmin=vmin, vmax=vmax)
        else: ax.plot_surface(X, Y, U[t], cmap="magma")
        
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_zlabel("z")
        ax.set_title(title)

    # create and save animation
    ani = animation.FuncAnimation(fig, update, U.shape[0], interval=interval)
    plt.close(fig)
    ani.save(filename)

In [3]:
def explicit_heat_solver_2d(nu, u0, x_span, y_span, tf, nx, ny, nt):

    # define space and time domains
    x_vals, dx = np.linspace(*x_span, nx+1, retstep=True)
    y_vals, dy = np.linspace(*y_span, ny+1, retstep=True)
    t_vals, dt = np.linspace(0, tf, nt+1, retstep=True)
    X, Y = np.meshgrid(x_vals, y_vals)

    # compute CFL values
    lam_x = nu * dt / dx**2
    lam_y = nu * dt / dy**2

    # compute the initial state
    U_init = np.array(
        [[u0(X[i, j], Y[i, j]) for i in range(X.shape[0])] for j in range(X.shape[1])]
    )
    
    # build stiffness matrix
    T = sparse.diags(
        diagonals=[lam_x, 1. - 2*lam_x - 2*lam_y, lam_x],
        offsets=[-1, 0, 1],
        shape=(nx-1, nx-1),
    )
    A = sparse.block_diag([T] * (ny-1))
    A.setdiag(lam_y, -(nx-1))
    A.setdiag(lam_y, nx-1)

    # initialize solution matrix -- homogenous Dirichlet BCs
    U = np.zeros((nt+1, nx+1, ny+1))
    sl = np.s_[1:-1]                    # interior slice 
    U[0, sl, sl] = U_init[sl, sl]       # set initial state on interior

    # solver loop 
    for t in range(1, nt+1):
        # this only works for homogenous Dirichlet BCs
        # if non-homoegenous, there should be an additive term on the right side
        U[t, sl, sl] = (A @ U[t-1, sl, sl].flatten()).reshape((nx-1, ny-1))

    return U, X, Y, t_vals

In [ ]:
# model parameters 
nu = 1.             # diffusion coefficient
x0, xf = 0, 1       # space domain (x)
y0, yf = 0, 1       # space domain (y)
tf = 0.25           # time domain
def u0(x, y):       # initial state
    return (
        np.sin(np.pi * (x - x0) / (xf - x0)) * 
        np.sin(np.pi * (y - y0) / (yf - y0))
    )

# # initial state (Gaussian bump)
# x_mid = (xf - x0) / 2.
# y_mid = (yf - y0) / 2.
# u0 = lambda x, y : np.exp((x - x_mid)**2 + (y - y_mid)**2)

# system parameters 
nx = 10 
ny = 10
nt = 100 

# test solver 
U, X, Y, t_vals = explicit_heat_solver_2d(nu, u0, (x0, xf), (y0, yf), tf, nx, ny, nt)

animate_heat_2d(
    U, X, Y,
    "Explicit Solver for 2D Heat Equation",
    "../animations/explicit_2d.mp4",
    interval=50
)

<video src="../animations/explicit_2d.mp4" controls>

In [ ]:
# model parameters 
nu = 1.             # diffusion coefficient
x0, xf = 0, 1       # space domain (x)
y0, yf = 0, 1       # space domain (y)
tf = 0.25           # time domain
def u0(x, y):       # initial state
    return (
        np.sin(np.pi * (x - x0) / (xf - x0)) * 
        np.sin(np.pi * (y - y0) / (yf - y0))
    )

# system parameters 
nx = 50 
ny = 50
nt = 100 

# test solver with broken CFL condition
U, X, Y, t_vals = explicit_heat_solver_2d(nu, u0, (x0, xf), (y0, yf), tf, nx, ny, nt)

animate_heat_2d(
    U, X, Y,
    "Explicit Solver for 2D Heat Equation, Unstable",
    "../animations/explicit_2d_unstable.mp4",
    interval=40, 
    set_vlim=False
)

<video src="../animations/explicit_2d_unstable.mp4" controls> 

Develop a 2D Crank-Nicolson solver

In [6]:
def crank_nicolson_heat_solver_2d(nu, u0, x_span, y_span, tf, nx, ny, nt):
    # define space and time domains
    x_vals, dx = np.linspace(*x_span, nx+1, retstep=True)
    y_vals, dy = np.linspace(*y_span, ny+1, retstep=True)
    t_vals, dt = np.linspace(0, tf, nt+1, retstep=True)
    X, Y = np.meshgrid(x_vals, y_vals)

    # compute CFL values
    lam_x = nu * dt / dx**2
    lam_y = nu * dt / dy**2

    # compute the initial state
    U_init = np.array(
        [[u0(X[i, j], Y[i, j]) for i in range(X.shape[0])] for j in range(X.shape[1])]
    )
    
    # construct stiffness matrices
    # Q (explicit side)
    T_Q = sparse.diags(
        diagonals=[lam_x/2., 1.-lam_x-lam_y, lam_x/2.],
        offsets=[-1, 0, 1],
        shape=(nx-1, nx-1)
    )
    Q = sparse.block_diag([T_Q] * (ny-1))
    Q.setdiag(lam_y/2., -(nx-1))
    Q.setdiag(lam_y/2., nx-1)
    Q = Q.tocsr()

    # P (implicit side)
    T_P = sparse.diags(
        diagonals=[-lam_x/2., 1+lam_x+lam_y, -lam_x/2.],
        offsets=[-1, 0, 1],
        shape=(nx-1, nx-1)
    )
    P = sparse.block_diag([T_P] * (ny-1))
    P.setdiag(-lam_y/2., -(nx-1))
    P.setdiag(-lam_y/2., nx-1)
    P = P.tocsr()

    # initialize solution matrix -- homogenous Dirichlet BCs
    U = np.zeros((nt+1, nx+1, ny+1))
    sl = np.s_[1:-1]                    # interior slice 
    U[0, sl, sl] = U_init[sl, sl]       # set initial state on interior

    # solver loop 
    for t in range(1, nt+1):
        U[t, sl, sl] = (sparse.linalg.spsolve(P, Q @ U[t-1, sl, sl].flatten())).reshape((nx-1, ny-1))

    return U, X, Y, t_vals

In [ ]:
# model parameters 
nu = 1.             # diffusion coefficient
x0, xf = 0, 1       # space domain (x)
y0, yf = 0, 1       # space domain (y)
tf = 0.25           # time domain
def u0(x, y):       # initial state
    return (
        np.sin(np.pi * (x - x0) / (xf - x0)) * 
        np.sin(np.pi * (y - y0) / (yf - y0))
    )

# # initial state (Gaussian bump)
# x_mid = (xf - x0) / 2.
# y_mid = (yf - y0) / 2.
# u0 = lambda x, y : np.exp((x - x_mid)**2 + (y - y_mid)**2)

# system parameters 
nx = 50 
ny = 50
nt = 100 

# test solver 
U, X, Y, t_vals = crank_nicolson_heat_solver_2d(nu, u0, (x0, xf), (y0, yf), tf, nx, ny, nt)

animate_heat_2d(
    U, X, Y,
    "Crank-Nicolson Solver for 2D Heat Equation",
    "../animations/crank_nicolson_2d.mp4",
    interval=40
)

<video src="../animations/crank_nicolson_2d.mp4" controls>

2D Exponential Solver

In [3]:
def exponential_heat_solver_2d(nu, u0, x_span, y_span, tf, nx, ny, nt):
    # define space and time domains
    x_vals, dx = np.linspace(*x_span, nx+1, retstep=True)
    y_vals, dy = np.linspace(*y_span, ny+1, retstep=True)
    t_vals, dt = np.linspace(0, tf, nt+1, retstep=True)
    X, Y = np.meshgrid(x_vals, y_vals)

    # pre-compute these values
    dx2_inv = 1. / dx**2
    dy2_inv = 1. / dy**2

    # compute the initial state
    U_init = np.array(
        [[u0(X[i, j], Y[i, j]) for i in range(X.shape[0])] for j in range(X.shape[1])]
    )

    # define differential operator
    T = sparse.diags(
        diagonals=[dx2_inv, -2.*(dx2_inv+dy2_inv), dx2_inv],
        offsets=[-1, 0, 1],
        shape=(nx-1, nx-1)
    )
    L = sparse.block_diag([T] * (ny-1))
    L.setdiag(dy2_inv, -(ny-1))
    L.setdiag(dy2_inv, ny-1)
    # L = L.toarray()
    L = L.tocsr()

    # exponentiate the linear operator via diagonalization
    # lam, V = sparse.linalg.eigs(dt * nu * L, k=(nx-1)*(ny-1) - 2)
    # lam, V = np.linalg.eig(dt * nu * L)
    # exp_operator = V @ np.diag(np.exp(lam)) @ np.linalg.inv(V)
    exp_operator = sparse.linalg.expm(dt * nu * L)

    # initialize solution matrix -- homogenous Dirichlet BCs
    U = np.zeros((nt+1, nx+1, ny+1))
    sl = np.s_[1:-1]                    # interior slice 
    U[0, sl, sl] = U_init[sl, sl]       # set initial state on interior

    # solver loop
    for t in range(1, nt+1):
        U[t, sl, sl] = (exp_operator @ U[t-1, sl, sl].flatten()).reshape((nx-1, ny-1))

    return U, X, Y, t_vals

In [4]:
# model parameters 
nu = 1.             # diffusion coefficient
x0, xf = 0, 1       # space domain (x)
y0, yf = 0, 1       # space domain (y)
tf = 0.25           # time domain
def u0(x, y):       # initial state
    return (
        np.sin(np.pi * (x - x0) / (xf - x0)) * 
        np.sin(np.pi * (y - y0) / (yf - y0))
    )

# # initial state (Gaussian bump)
# x_mid = (xf - x0) / 2.
# y_mid = (yf - y0) / 2.
# u0 = lambda x, y : np.exp((x - x_mid)**2 + (y - y_mid)**2)

# system parameters 
nx = 10 
ny = 10
nt = 100 

# test solver 
U, X, Y, t_vals = exponential_heat_solver_2d(nu, u0, (x0, xf), (y0, yf), tf, nx, ny, nt)



/Users/mckayladavis/opt/anaconda3/envs/acme/lib/python3.10/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:412: SparseEfficiencyWarning: splu converted its input to CSC format
  warn('splu converted its input to CSC format', SparseEfficiencyWarning)
/Users/mckayladavis/opt/anaconda3/envs/acme/lib/python3.10/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:302: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  warn('spsolve is more efficient when sparse b '


In [9]:
animate_heat_2d(
    U, X, Y,
    "Exponential Solver for 2D Heat Equation",
    "../animations/exponential_2d.mp4",
    interval=40
)

<video src="../animations/exponential_2d.mp4" controls> 

Test a 2D heat solver on Chebyshev points.

In [5]:
def exponential_heat_solver_2d_chebyshev(nu, u0, tf, n_cheb_x, n_cheb_y, nt):
    # define space and time domains using Chebyshev nodes on -1 to 1,
    # also get the differentiation matrices in X and Y directions
    Dx, x_vals = cheb(n_cheb_x)
    Dy, y_vals = cheb(n_cheb_y)
    X, Y = np.meshgrid(x_vals, y_vals)
    t_vals, dt = np.linspace(0, tf, nt+1, retstep=True)

    # compute the second-order Chebyshev differentiation matrix
    Dx2 = Dx @ Dx 
    Dy2 = Dy @ Dy

    # compute the initial state
    U_init = np.array(
        [[u0(X[i, j], Y[i, j]) for i in range(X.shape[0])] for j in range(X.shape[1])]
    )

    # define differential operator
    sl = np.s_[1:-1]                    # interior slice 
    L = np.kron(Dx2[sl, sl], np.eye(n_cheb_y-1)) + np.kron(np.eye(n_cheb_x-1), Dy2[sl, sl])

    # exponentiate the linear operator via diagonalization
    # lam, V = sparse.linalg.eigs(dt * nu * L, k=(nx-1)*(ny-1) - 2)
    lam, V = np.linalg.eig(dt * nu * L)
    exp_operator = V @ np.diag(np.exp(lam)) @ np.linalg.inv(V)
    # exp_operator = sparse.linalg.expm(dt * nu * L)

    # initialize solution matrix -- homogenous Dirichlet BCs
    U = np.zeros((nt+1, n_cheb_x+1, n_cheb_y+1))
    U[0, sl, sl] = U_init[sl, sl]       # set initial state on interior

    # solver loop
    for t in range(1, nt+1):
        U[t, sl, sl] = (exp_operator @ U[t-1, sl, sl].flatten()).reshape((n_cheb_x-1, n_cheb_y-1))

    return U, X, Y, t_vals

In [41]:
# model parameters 
nu = 1.             # diffusion coefficient
x0, xf = -1, 1
y0, yf = -1, 1
n_cheb_x = 16 
n_cheb_y = 16 
tf = 0.5           # time domain
nt = 100 

def u0(x, y):       # initial state
    return (
        np.sin(np.pi * (x - x0) / (xf - x0)) * 
        np.sin(np.pi * (y - y0) / (yf - y0))
    )

# test solver 
U, X, Y, t_vals = exponential_heat_solver_2d_chebyshev(nu, u0, tf, n_cheb_x, n_cheb_y, nt)

(225, 225)


/tmp/ipykernel_6181/22706617.py:35: ComplexWarning: Casting complex values to real discards the imaginary part
  U[t, sl, sl] = (exp_operator @ U[t-1, sl, sl].flatten()).reshape((n_cheb_x-1, n_cheb_y-1))


In [ ]:
animate_heat_2d(
    U, X, Y,
    "Exponential Solver for 2D Heat Equation on Chebyshev Nodes",
    "../animations/exponential_cheb_2d.mp4",
    interval=40, 
    x_lim=[-1, 1],
    y_lim=[-1, 1],
)

<video src="../animations/exponential_cheb_2d.mp4" controls> 

Compare error between Chebyshev grid points and evenly-spaced grid points

In [6]:
def compare_errors(max_pow=4):

    # values to use for number of Chebyshev nodes
    N_vals = 2 ** np.arange(2, max_pow+1)

    # model parameters 
    nu = 1.             # diffusion coefficient
    x0, xf = -1, 1      # x domain
    y0, yf = -1, 1      # y domain
    tf = 0.5            # time domain
    nt = 100            # number of time steps 

    # initial condition 
    u0 = lambda x, y : np.sin(np.pi*(x+1.)/2.) * np.sin(np.pi*(y+1.)/2.)

    # analytic solution
    u_analytic = lambda x, y, t : np.sin(np.pi*(x+1.)/2.) * np.sin(np.pi*(y+1.)/2.) * np.exp(-nu*t*np.pi**2/2.)

    # initialize arrays to store relative error and runtime
    rel_errors_even = np.empty_like(N_vals, dtype=np.float64)
    runtimes_even = np.empty_like(N_vals, dtype=np.float64)

    rel_errors_cheb = np.empty_like(N_vals, dtype=np.float64)
    runtimes_cheb = np.empty_like(N_vals, dtype=np.float64)

    for i, N in enumerate(N_vals):
        # approximate on even grid points
        start = time()
        U_even, X_even, Y_even, t_vals = exponential_heat_solver_2d(nu, u0, (x0, xf), (y0, yf), tf, N, N, nt)
        end = time()
        runtimes_even[i] = end - start

        # approximate on Chebyshev points
        start = time()
        U_cheb, X_cheb, Y_cheb, t_vals = exponential_heat_solver_2d_chebyshev(nu, u0, tf, N, N, nt)
        end = time()
        runtimes_cheb[i] = end - start 

        # compute the error for each solution 
        even_analytic = np.array([u_analytic(X_even, Y_even, t) for t in t_vals])
        cheb_analytic = np.array([u_analytic(X_cheb, Y_cheb, t) for t in t_vals])

        # compute relative error for each approximation
        rel_error_even = np.linalg.norm(even_analytic - U_even) / np.linalg.norm(even_analytic)
        rel_error_cheb = np.linalg.norm(cheb_analytic - U_cheb) / np.linalg.norm(cheb_analytic)   

        rel_errors_even[i] = rel_error_even
        rel_errors_cheb[i] = rel_error_cheb
    

    fig = plt.figure(figsize=(10, 7))

    ax1 = fig.add_subplot(211)
    ax1.loglog(N_vals, rel_errors_even, marker='o', label="Even Grid")
    ax1.loglog(N_vals, rel_errors_cheb, marker='o', label="Chebyshev Grid")
    ax1.set_title("Relative Errors for Evenly Spaced vs. Chebyshev Grids")
    ax1.set_xlabel("Number of Grid Points")
    ax1.set_ylabel("Relative Error")
    ax1.legend()

    ax2 = fig.add_subplot(212)
    ax2.loglog(N_vals, runtimes_even, marker='o', label="Even Grid")
    ax2.loglog(N_vals, runtimes_cheb, marker='o', label="Chebyshev Grid")
    ax2.set_title("Runtimes for Evenly Spaced vs. Chebyshev Grids")
    ax2.set_xlabel("Number of Grid Points")
    ax2.set_ylabel("Runtime (s)")
    ax2.legend()

    plt.tight_layout()
    plt.savefig("../plots/cheb_even_comparison.png", dpi=300)
    plt.show()

In [ ]:
compare_errors(max_pow=10)

/var/folders/sv/3h00w8hd0kj7ncygzkyfbr580000gn/T/ipykernel_21376/3204038036.py:34: ComplexWarning: Casting complex values to real discards the imaginary part
  U[t, sl, sl] = (exp_operator @ U[t-1, sl, sl].flatten()).reshape((n_cheb_x-1, n_cheb_y-1))


Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
